# Crime Data Pipeline — Police Force Analytics Unit
**Forces:** Metropolitan Police, West Midlands Police, West Yorkshire Police, Thames Valley Police  
**Period:** April 2023 – March 2026  
**Author:** Izzam Golamaully
**Date:** May 2026  

## Pipeline Stages
1. Ingestion Layer
2. Cleaning & Validation Layer
3. Feature Engineering & Transformation
4. Enrichment & Aggregation
5. Export & Final Validation

In [3]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

print("Libraries loaded ✓")

Libraries loaded ✓


In [4]:
# All paths relative to the notebook location
BASE_DIR = Path("..") 
RAW_POLICE_DIR = BASE_DIR / "data" / "raw" / "police"
ENRICHMENT_DIR = BASE_DIR / "data" / "raw" / "enrichment"
OUTPUT_DIR     = BASE_DIR / "data" / "outputs"

# Create output dir if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify paths exist
for p in [RAW_POLICE_DIR, ENRICHMENT_DIR]:
    status = "✓" if p.exists() else "✗ NOT FOUND"
    print(f"{p}: {status}")

..\data\raw\police: ✓
..\data\raw\enrichment: ✓


In [5]:
# Get all CSV files in the police data folder
all_files = glob.glob(str(RAW_POLICE_DIR / "**" / "*.csv"), recursive=True)

print(f"Total CSV files found: {len(all_files)}")
print("\nFirst 10 files:")
for f in sorted(all_files)[:10]:
    print(f)

Total CSV files found: 144

First 10 files:
..\data\raw\police\2023-04\2023-04-metropolitan-street.csv
..\data\raw\police\2023-04\2023-04-thames-valley-street.csv
..\data\raw\police\2023-04\2023-04-west-midlands-street.csv
..\data\raw\police\2023-04\2023-04-west-yorkshire-street.csv
..\data\raw\police\2023-05\2023-05-metropolitan-street.csv
..\data\raw\police\2023-05\2023-05-thames-valley-street.csv
..\data\raw\police\2023-05\2023-05-west-midlands-street.csv
..\data\raw\police\2023-05\2023-05-west-yorkshire-street.csv
..\data\raw\police\2023-06\2023-06-metropolitan-street.csv
..\data\raw\police\2023-06\2023-06-thames-valley-street.csv


In [6]:
# Load just one file to inspect structure
sample_file = sorted(all_files)[0]
print(f"Inspecting: {sample_file}\n")

sample_df = pd.read_csv(sample_file)

print(f"Shape: {sample_df.shape}")
print(f"\nColumns:\n{list(sample_df.columns)}")
print(f"\nData types:\n{sample_df.dtypes}")
print(f"\nFirst 3 rows:")
sample_df.head(3)

Inspecting: ..\data\raw\police\2023-04\2023-04-metropolitan-street.csv

Shape: (87547, 12)

Columns:
['Crime ID', 'Month', 'Reported by', 'Falls within', 'Longitude', 'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type', 'Last outcome category', 'Context']

Data types:
Crime ID                  object
Month                     object
Reported by               object
Falls within              object
Longitude                float64
Latitude                 float64
Location                  object
LSOA code                 object
LSOA name                 object
Crime type                object
Last outcome category     object
Context                  float64
dtype: object

First 3 rows:


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,7c85d5925620a55fd890ea37dd748a34d2c77773f5d740...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.254186,50.834677,On or near Kingsland Close,E01031372,Adur 004E,Vehicle crime,Investigation complete; no suspect identified,NaN
1,43ccf73d9d380e4fcd128d7384d49ccf8d5ed54887a940...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.559343,50.852819,On or near School Lane,E01031392,Arun 001C,Violence and sexual offences,Investigation complete; no suspect identified,NaN
2,dd453aa29e0058e3fd98c507bf9429b10d654b653571f7...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.405658,50.865992,On or near Southview Road,E01031425,Arun 002C,Violence and sexual offences,Investigation complete; no suspect identified,NaN


In [7]:
# Check null % on the sample file before ingesting everything
print("Null % per column:")
null_pct = (sample_df.isnull().sum() / len(sample_df) * 100).round(2)
print(null_pct)

print(f"\nUnique crime types ({sample_df['Crime type'].nunique()}):")
print(sorted(sample_df['Crime type'].unique()))

Null % per column:
Crime ID                  19.80
Month                      0.00
Reported by                0.00
Falls within               0.00
Longitude                  2.31
Latitude                   2.31
Location                   0.00
LSOA code                  2.31
LSOA name                  2.31
Crime type                 0.00
Last outcome category     19.80
Context                  100.00
dtype: float64

Unique crime types (14):
['Anti-social behaviour', 'Bicycle theft', 'Burglary', 'Criminal damage and arson', 'Drugs', 'Other crime', 'Other theft', 'Possession of weapons', 'Public order', 'Robbery', 'Shoplifting', 'Theft from the person', 'Vehicle crime', 'Violence and sexual offences']


---
## Stage 1: Ingestion Layer

Batch ingestion — one force at a time. Each CSV is loaded individually and 
unioned into a single DataFrame. The `force_name` field is extracted from 
the filename. Raw row counts are logged per force for validation.